In [3]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
print("Num GPUs:", len(tf.config.list_physical_devices('GPU')))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Num GPUs: 1


## 1. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

## 3. Load Datasets

In [6]:
# base_path  = '/content/drive/MyDrive/Colab Notebooks/pollinators/'
# BATCH_SIZE = 32
# IMG_SIZE   = (224, 224)

# # Load train and val in one call using a workaround
# import numpy as np

# full_ds = tf.keras.utils.image_dataset_from_directory(
#     base_path + 'train',
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     seed=42
# )

# class_names = full_ds.class_names
# num_classes = len(class_names)
# print(f'Number of classes: {num_classes}')

# # Manually split 80/20
# total_batches = len(full_ds)
# train_size    = int(0.8 * total_batches)

# train_ds_raw = full_ds.take(train_size)
# val_ds_raw   = full_ds.skip(train_size)

# print(f'Train batches: {train_size}')
# print(f'Val batches:   {total_batches - train_size}')

# # Test set stays separate
# test_ds_raw = tf.keras.utils.image_dataset_from_directory(
#     base_path + 'test',
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
# )

from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf

# -----------------------------
# Dataset paths and parameters
# -----------------------------
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/archive'
TRAIN_DIR = os.path.join(BASE_PATH, 'train')
VALID_DIR = os.path.join(BASE_PATH, 'valid')
TEST_DIR  = os.path.join(BASE_PATH, 'test')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
SEED = 42

# Check that folders exist
for folder in [TRAIN_DIR, VALID_DIR, TEST_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Missing folder: {folder}")

print("Dataset folders found:")
print("Train:", TRAIN_DIR)
print("Valid:", VALID_DIR)
print("Test :", TEST_DIR)

# Load datasets

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Class information

class_names = train_ds.class_names
num_classes = len(class_names)

print(f"\nNumber of classes: {num_classes}")
print("First 10 classes:", class_names[:10])

# Improve performance

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset folders found:
Train: /content/drive/MyDrive/Colab Notebooks/archive/train
Valid: /content/drive/MyDrive/Colab Notebooks/archive/valid
Test : /content/drive/MyDrive/Colab Notebooks/archive/test
Found 12594 files belonging to 100 classes.
Found 500 files belonging to 100 classes.
Found 500 files belonging to 100 classes.

Number of classes: 100
First 10 classes: ['ADONIS', 'AFRICAN GIANT SWALLOWTAIL', 'AMERICAN SNOOT', 'AN 88', 'APPOLLO', 'ARCIGERA FLOWER MOTH', 'ATALA', 'ATLAS MOTH', 'BANDED ORANGE HELICONIAN', 'BANDED PEACOCK']


In [ ]:
#added recently - RT for hyperparameters

import os
import pandas as pd


# Hyperparameters
#BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/archive'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
SEED = 42

EPOCHS = 18
LEARNING_RATE = 1e-4

# Data augmentation
ROTATION = 0.2
ZOOM = 0.2
TRANSLATE_H = 0.1
TRANSLATE_W = 0.1
CONTRAST = 0.2

# CNN architecture
FILTERS_1 = 32
FILTERS_2 = 64
FILTERS_3 = 128
FILTERS_4 = 256
FILTERS_5 = 512

DENSE_UNITS = 256
DROPOUT_RATE = 0.6
L2_WEIGHT = 0.005

# Callbacks
EARLY_STOP_PATIENCE = 7
REDUCE_LR_FACTOR = 0.5
REDUCE_LR_PATIENCE = 3
MIN_LR = 1e-7

# Optional: name this experiment
RUN_NAME = "baseline_run"

print("Current hyperparameters loaded.")

## 4. Normalization + Performance Pipeline

In [7]:
AUTOTUNE            = tf.data.AUTOTUNE
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)

## 5. Data Augmentation

In [8]:
# data_augmentation = tf.keras.Sequential([
#     layers.RandomFlip('horizontal'),
#     layers.RandomFlip('vertical'),
#     layers.RandomRotation(0.2),
#     layers.RandomZoom(0.2),
#     layers.RandomTranslation(0.1, 0.1),
#     layers.RandomContrast(0.2),
# ], name='augmentation')

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomRotation(ROTATION),
    layers.RandomZoom(ZOOM),
    layers.RandomTranslation(TRANSLATE_H, TRANSLATE_W),
    layers.RandomContrast(CONTRAST),
], name="augmentation")

## 6. Build Model

In [9]:
# model = models.Sequential([
#     data_augmentation,

#     # Block 1
#     layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2, 2),

#     # Block 2
#     layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2, 2),

#     # Block 3
#     layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2, 2),

#     # Block 4
#     layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2, 2),

#     # Block 5
#     layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2, 2),

#     layers.GlobalAveragePooling2D(),
#     layers.Dense(256, activation='relu',
#                  kernel_regularizer=tf.keras.regularizers.l2(0.005)),
#     layers.Dropout(0.6),
#     layers.Dense(num_classes, activation='softmax')
# ])

# model.build(input_shape=(None, 224, 224, 3))
# model.summary()


#added recently - RT
model = models.Sequential([
    data_augmentation,

    # Block 1
    layers.Conv2D(FILTERS_1, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(FILTERS_2, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(FILTERS_3, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 4
    layers.Conv2D(FILTERS_4, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 5
    layers.Conv2D(FILTERS_5, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(
        DENSE_UNITS,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(L2_WEIGHT)
    ),
    layers.Dropout(DROPOUT_RATE),
    layers.Dense(num_classes, activation='softmax')
])

model.build(input_shape=(None, 224, 224, 3))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,729,572 (6.60 MB)

 Trainable params: 1,727,588 (6.59 MB)

 Non-trainable params: 1,984 (7.75 KB)

## 7. Compile

In [10]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
# added recently - RT
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## 8. Callbacks

In [11]:
# callbacks = [
#     EarlyStopping(
#         monitor='val_accuracy',
#         patience=7,
#         restore_best_weights=True,
#         verbose=1
#     ),
#     ReduceLROnPlateau(
#         monitor='val_loss',
#         factor=0.5,
#         patience=3,
#         min_lr=1e-7,
#         verbose=1
#     )
# ]

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=MIN_LR,
        verbose=1
    )
]

## 9. Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=18,
    callbacks=callbacks
)

Epoch 1/18


## 10. Evaluate on Test Set

In [ ]:
print('\n=== TEST SET EVALUATION ===')
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss:     {test_loss:.4f}')

In [ ]:
#added recently - RT

print("\n--- TEST SET EVALUATION ---")
test_loss, test_acc = model.evaluate(test_ds)

print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")

run_result = {
    "run_name": RUN_NAME,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "dropout_rate": DROPOUT_RATE,
    "l2_weight": L2_WEIGHT,
    "dense_units": DENSE_UNITS,
    "filters": f"{FILTERS_1}-{FILTERS_2}-{FILTERS_3}-{FILTERS_4}-{FILTERS_5}",
    "rotation": ROTATION,
    "zoom": ZOOM,
    "translate_h": TRANSLATE_H,
    "translate_w": TRANSLATE_W,
    "contrast": CONTRAST,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "reduce_lr_factor": REDUCE_LR_FACTOR,
    "reduce_lr_patience": REDUCE_LR_PATIENCE,
    "min_lr": MIN_LR,
    "best_val_accuracy": max(history.history["val_accuracy"]),
    "best_val_loss": min(history.history["val_loss"]),
    "final_test_accuracy": test_acc,
    "final_test_loss": test_loss
}

if "results_log" not in globals():
    results_log = []

results_log.append(run_result)

results_df = pd.DataFrame(results_log)
results_df

## 11. Save Model

In [ ]:
# save_path = base_path + 'pollinator_model.keras'
# model.save(save_path)
# print(f'Model saved to: {save_path}')

#added recently - RT

save_path = os.path.join(BASE_PATH, "butterfly_model.keras")
model.save(save_path)
print(f"Model saved to: {save_path}")

## 12. Plot Training History

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ax1.plot(history.history['accuracy'],     label='Train Accuracy')
# ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
# ax1.set_title('Accuracy')
# ax1.set_xlabel('Epoch')
# ax1.legend()

# ax2.plot(history.history['loss'],     label='Train Loss')
# ax2.plot(history.history['val_loss'], label='Val Loss')
# ax2.set_title('Loss')
# ax2.set_xlabel('Epoch')
# ax2.legend()

# plt.tight_layout()
# plt.savefig(base_path + 'training_history.png')
# plt.show()


# added recently - RT
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], marker='o', label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], marker='o', label='Val Accuracy')
ax1.set_title('Training vs Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], marker='o', label='Train Loss')
ax2.plot(history.history['val_loss'], marker='o', label='Val Loss')
ax2.set_title('Training vs Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plot_path = os.path.join(BASE_PATH, f"{RUN_NAME}_training_history.png")
plt.savefig(plot_path, bbox_inches='tight')
plt.show()

print(f"Training history plot saved to: {plot_path}")

In [ ]:
#Compare Hyperparameter Runs

if "results_df" in globals() and len(results_df) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(results_df["run_name"], results_df["final_test_accuracy"])
    plt.title("Test Accuracy by Hyperparameter Run")
    plt.xlabel("Run Name")
    plt.ylabel("Test Accuracy")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    display(results_df.sort_values("final_test_accuracy", ascending=False))